# IEEE-CIS Fraud Detection - Advanced Feature Engineering

## Objective
Engineer advanced features to boost baseline performance.

# 1. Setup and Load data
Import libraries and load processed data.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix, precision_recall_curve, roc_curve)
import xgboost as xgb
import warnings
import gc
from datetime import datetime
import os

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [2]:
# Import data
train = pd.read_csv('../../data/interim/train.csv')

In [3]:
train.shape

(590540, 434)

# 2. Part A: Missing Value Features
We create binary "is_missing" flags for features with 25-75% missing values because our EDA showed that when ertain features are missing, fraud rate increases.

In [4]:
# Identify features for missing value flags
# Calculate missing percentages
missing = train.isnull().sum()
missing_pct = (missing / len(train)) * 100

missing_df = pd.DataFrame({
    'feature': missing.index,
    'missing_count': missing.values,
    'missing_pct': missing_pct.values
}).sort_values('missing_pct', ascending=False)

# Filter for missing_pct values 25-75%
features_for_flags = missing_df[
    (missing_df['missing_pct'] >= 25) &
    (missing_df['missing_pct'] <= 75)
]['feature'].tolist()

features_for_flags = [f for f in features_for_flags if f not in ['isFraud','TransactionID' ]]
print(len(features_for_flags))



44


In [5]:
# Create 'is_missing' flag
# Track new features
new_features = []
feature_creation_log = []

# Create flags
for feature in tqdm(features_for_flags, desc='Creating flags'):
    flag_name = f"{feature}_is_missing"
    train[flag_name] = train[feature].isna().astype(np.int8)

    # Log
    new_features.append(flag_name)
    missing_rate = train[flag_name].mean()
    feature_creation_log.append({
        'original_feature': feature,
        'flag_feature': flag_name,
        'missing_rate': missing_rate
    })

print(len(new_features))

# Test
test_flag = new_features[0]
print(test_flag)
fraud_when_missing = train[train[test_flag] == 1]['isFraud'].mean() * 100
fraud_when_present = train[train[test_flag] == 0]['isFraud'].mean() * 100

print(fraud_when_missing)
print(fraud_when_present)


Creating flags: 100%|██████████| 44/44 [00:00<00:00, 62.50it/s]


44
dist1_is_missing
4.5158414970292755
1.9956435793158152


In [6]:
# Analysing most informative missing flags
flag_analysis = []

for flag in tqdm(new_features[:50], desc='Analysing flags'):
    fraud_when_1 = train[train[flag] == 1]['isFraud'].mean() * 100
    fraud_when_0 = train[train[flag] == 0]['isFraud'].mean() * 100
    difference = fraud_when_1 - fraud_when_0

    flag_analysis.append({
        'flag': flag,
        'fraud_when_missing': fraud_when_1,
        'fraud_when_present': fraud_when_0,
        'difference': difference,
        'abs_difference': abs(difference)
    })

# Convert to dataframe
flag_analysis_df = pd.DataFrame(flag_analysis).sort_values('abs_difference', ascending=False)
print(flag_analysis_df.head())


Analysing flags: 100%|██████████| 44/44 [02:02<00:00,  2.79s/it]

             flag  fraud_when_missing  fraud_when_present  difference  \
24  M6_is_missing            7.068375            2.063726    5.004649   
22  M2_is_missing            5.282553            1.985349    3.297203   
21  M3_is_missing            5.282553            1.985349    3.297203   
20  M1_is_missing            5.282553            1.985349    3.297203   
15  V9_is_missing            5.212201            1.961748    3.250453   

    abs_difference  
24        5.004649  
22        3.297203  
21        3.297203  
20        3.297203  
15        3.250453  


# 3. Part B: Temporal Features
Here we will create time features, velocity features (transaction frequency and timing patterns), and time bases risk indicators.

In [7]:
# 1. Create hour of day
train['transaction_hour'] = (train['TransactionDT'] // 3600) % 24

# 2. Day number
train['transaction_day'] = (train['TransactionDT'] // (3600 * 24))

# 3. Day of week
train['transaction_weekday'] = (train['transaction_day'] % 7)

# 4. Is weekend?
train['is_weekend'] = (train['transaction_weekday'] >= 5).astype(np.int8)

# 5. Is night 
train['is_night'] = ((train['transaction_hour'] >= 0) & (train['transaction_hour'] < 6)).astype(np.int8)

# 6. Time period bins
def get_time_period(hour):
    if 0 <= hour < 6:
        return 0  # Night
    elif 6 <= hour < 12:
        return 1  # Morning
    elif 12 <= hour < 18:
        return 2  # Afternoon
    else:
        return 3  # Evening

train['time_period'] = train['transaction_hour'].apply(get_time_period).astype(np.int8)

# Fraud rate per hour
hourly_fraud = train.groupby('is_night')['isFraud'].mean() * 100
print(hourly_fraud.sort_values(ascending=False).head())

is_night
1    3.829924
0    3.393588
Name: isFraud, dtype: float64


In [8]:
# Velocity Features
# Sort time for velocity calculations
train_sorted = train.sort_values('TransactionDT').reset_index(drop=True)
train = train_sorted
del train_sorted
gc.collect(0)

0

In [9]:
# Velocity Features - Card Level
# Time since last transaction(by card)
train['time_since_last_txn'] = train.groupby('card1')['TransactionDT'].diff()
train['time_since_last_txn_hrs'] = train['time_since_last_txn'] / 3600

# Fill mssing with median
median_time = train['time_since_last_txn_hrs'].median()
train['time_since_last_txn_hrs'] = train['time_since_last_txn_hrs'].fillna(median_time)

# Transaction count in last one hour (by card)
def count_txns_in_window(group, window_seconds=3600):
    result = np.zeros(len(group), dtype=np.int16)
    times = group['TransactionDT'].values

    for i in range(len(times)):
        current_time=times[i]
        # Count transactions in [current-window, current_time]
        window_start = current_time - window_seconds
        count = np.sum((times >= window_start) & (times < current_time))
        result[i] = count
    
    return result

# Apply to each card
tqdm.pandas(desc="Card processed")
train['txn_count_1h'] = train.groupby('card1', group_keys=False).progress_apply(
    lambda x: pd.Series(count_txns_in_window(x), index=x.index)
).astype(np.int16)

print(f"Max trabsactions in 1h: {train['txn_count_1h'].max()}")
print(f"Mean transactions in 1h: {train['txn_count_1h'].mean():.2f}")

# Transaction count in last 24hrs
train['txn_count_24h'] = train.groupby('card1', group_keys=False).progress_apply(
    lambda x: pd.Series(count_txns_in_window(x, window_seconds=86400), index=x.index)
).astype(np.int16)

print(f"Max transactions in 24h: {train['txn_count_24h'].max()}")
print(f"Mean transactions in 24h: {train['txn_count_24h'].mean():.2f}")

# Transaction frequency (transactions per day by card)
card_freq = train.groupby('card1').agg({
    'TransactionID': 'count',
    'transaction_day': lambda x: x.max() - x.min() + 1
}).reset_index()
card_freq.columns = ['card1', 'total_txns', 'days_active']
card_freq['txn_frequency'] = card_freq['total_txns'] / card_freq['days_active']

# Merge back
train = train.merge(card_freq[['card1', 'txn_frequency']], on='card1', how='left')
# print(f"Mean frequency: {train['txn_frequency'].mean():.2f} transactions per day")

train.drop('time_since_last_txn', axis=1)

# Analyze velocity patterns
print(f"\n Velocity Pattern Analysis:")

# Moderate velocity (2 >= 5 txns in 1 hour) fraud rate
moderate_velocity_1h = ((train['txn_count_1h'] >= 2) & (train['txn_count_1h'] <= 5))
print(f"\n   Moderate velocity (2 >= 5 txns in 1h):")
print(f"      Transactions: {moderate_velocity_1h.sum():,} ({moderate_velocity_1h.mean()*100:.2f}%)")
print(f"      Fraud rate: {train[moderate_velocity_1h]['isFraud'].mean()*100:.2f}%")
print(f"      vs Normal: {train[~moderate_velocity_1h]['isFraud'].mean()*100:.2f}%")

# Quick succession (< 5 minutes since last)
quick_succession = train['time_since_last_txn_hrs'] < (5/60)
print(f"\n   Quick succession (<5 min since last):")
print(f"      Transactions: {quick_succession.sum():,} ({quick_succession.mean()*100:.2f}%)")
print(f"      Fraud rate: {train[quick_succession]['isFraud'].mean()*100:.2f}%")
print(f"      vs Normal: {train[~quick_succession]['isFraud'].mean()*100:.2f}%")

print(f"\n💡 Insight:")
if train[moderate_velocity_1h]['isFraud'].mean() > train[~moderate_velocity_1h]['isFraud'].mean():
    diff_pct = ((train[moderate_velocity_1h]['isFraud'].mean() / train[~moderate_velocity_1h]['isFraud'].mean()) - 1) * 100
    print(f"   → Moderate velocity transactions are {diff_pct:.0f}% MORE likely to be fraud!")
    print(f"   → Velocity is a STRONG fraud signal")
else:
    print(f"   → Velocity patterns not as expected (investigate further)")

print(f"\n Velocity features complete!")

Card processed: 100%|██████████| 13553/13553 [00:31<00:00, 428.53it/s] 


Max trabsactions in 1h: 191
Mean transactions in 1h: 1.46


Card processed: 100%|██████████| 13553/13553 [00:25<00:00, 528.08it/s] 


Max transactions in 24h: 880
Mean transactions in 24h: 18.80

 Velocity Pattern Analysis:

   Moderate velocity (2 >= 5 txns in 1h):
      Transactions: 109,393 (18.52%)
      Fraud rate: 4.40%
      vs Normal: 3.29%

   Quick succession (<5 min since last):
      Transactions: 76,685 (12.99%)
      Fraud rate: 5.66%
      vs Normal: 3.18%

💡 Insight:
   → Moderate velocity transactions are 34% MORE likely to be fraud!
   → Velocity is a STRONG fraud signal

 Velocity features complete!


In [10]:
# Temporal Indicators
# High velocity flag
train['is_moderate_velocity'] = (train['txn_count_1h'] > 3).astype(np.int8)
print("Moderate Velocity flag")
print(f"Flagged: {train['is_moderate_velocity'].sum():}, ({train['is_moderate_velocity'].mean() * 100:.2f}%)")

# Suspicious timing
train['is_suspicious_time'] = ((train['is_night'] ==1) & (train['is_weekend'] ==1)).astype(np.int8)
print('Suspicious timing')
print(f"Flagged: {train['is_suspicious_time'].sum():,} ({train['is_suspicious_time'].mean() * 100:.2f}%)")

# Rapid succession flag
train['is_rapid_succession'] = (train['time_since_last_txn_hrs'] < (10/60)).astype(np.int8)
print('Rapid succession flag')
print(f"Flagged: {train['is_rapid_succession'].sum():,} ({train['is_rapid_succession'].mean() * 100:.2f}%)")

# Peak fraud hour flag (hours with >4% fraud from EDA)
train['is_peak_fraud_hour'] = ((train['transaction_hour'] >= 2) & (train['transaction_hour'] <= 5)).astype(np.int8)
print('Peak fraud hour')
print(f"Flagged: {train['is_peak_fraud_hour'].sum():,} ({train['is_peak_fraud_hour'].mean() * 100:.2f}%)")

# Unusual frequency
freq_90th = train['txn_frequency'].quantile(0.90)
train['is_unusual_frequency'] = (train['txn_frequency'] > freq_90th).astype(np.int8)
print('Unusual frequency')
print(f"Threshold: {freq_90th:.2f} transactions/day")
print(f"Flagged: {train['is_unusual_frequency'].sum():,} ({train['is_unusual_frequency'].mean() * 100:.2f}%)")


print(f"\n📊 Risk Indicators Summary:")
print(f"   Risk flags created: 5")
print(f"   Total features now: {train.shape[1]}")

# Analyze combined risk
print(f"\n🚨 Combined Risk Analysis:")

# Multiple risk flags
train['risk_flag_count'] = (
    train['is_moderate_velocity'] +
    train['is_suspicious_time'] +
    train['is_rapid_succession'] +
    train['is_peak_fraud_hour'] +
    train['is_unusual_frequency']
)

print(f"\n   Fraud rate by number of risk flags:")
for flags in range(6):
    count = (train['risk_flag_count'] == flags).sum()
    if count > 100:  # Only show if sufficient data
        fraud_rate = train[train['risk_flag_count'] == flags]['isFraud'].mean() * 100
        pct = count / len(train) * 100
        print(f"      {flags} flags: {fraud_rate:5.2f}% fraud ({count:>7,} txns, {pct:5.2f}%)")

print(f"\n💡 Insight:")
fraud_0_flags = train[train['risk_flag_count'] == 0]['isFraud'].mean() * 100
fraud_3plus_flags = train[train['risk_flag_count'] >= 3]['isFraud'].mean() * 100

if fraud_3plus_flags > fraud_0_flags:
    multiplier = fraud_3plus_flags / fraud_0_flags
    print(f"   → Transactions with 3+ risk flags are {multiplier:.1f}× MORE likely to be fraud!")
    print(f"   → Combined risk indicators work! ✅")

print(f"\n✅ Temporal risk indicators complete!")



Moderate Velocity flag
Flagged: 59452, (10.07%)
Suspicious timing
Flagged: 42,655 (7.22%)
Rapid succession flag
Flagged: 113,235 (19.17%)
Peak fraud hour
Flagged: 72,074 (12.20%)
Unusual frequency
Threshold: 38.96 transactions/day
Flagged: 57,744 (9.78%)

📊 Risk Indicators Summary:
   Risk flags created: 5
   Total features now: 494

🚨 Combined Risk Analysis:

   Fraud rate by number of risk flags:
      0 flags:  3.00% fraud (371,833 txns, 62.96%)
      1 flags:  4.54% fraud (129,741 txns, 21.97%)
      2 flags:  4.27% fraud ( 56,576 txns,  9.58%)
      3 flags:  3.62% fraud ( 28,001 txns,  4.74%)
      4 flags:  4.27% fraud (  3,681 txns,  0.62%)
      5 flags:  2.82% fraud (    708 txns,  0.12%)

💡 Insight:
   → Transactions with 3+ risk flags are 1.2× MORE likely to be fraud!
   → Combined risk indicators work! ✅

✅ Temporal risk indicators complete!


# 4. Aggregatioon Features
In this section, we will create behavioral statistics aggregated by card, product, and email domain.
Undeerstanding normal behavior is crucial in fraud detection. Each card feature has a 'normal' behaviour and deviation from that normal behaviour is suspicious.

In [11]:
# Card level aggregations
card_aggs = train.groupby('card1').agg({
    'TransactionAmt': ['mean', 'std', 'min', 'max', 'median', 'count'],
    'isFraud': 'mean'
}).reset_index()

# Flatten column names
card_aggs.columns = [
    'card1',
    'card_amt_mean',
    'card_amt_std',
    'card_amt_min',
    'card_amt_max',
    'card_amt_median',
    'card_txn_count',
    'card_fraud_rate'
]

print(card_aggs.head())

train = train.merge(card_aggs, on='card1', how='left')
print(train.shape[1])

# Analyse Fraud Patterns
# Cards with high historical fraud rate
high_fraud_cards = card_aggs[card_aggs['card_fraud_rate'] > 0.5]
print(f"High risk cards: {len(high_fraud_cards)}")
if len(high_fraud_cards) > 0:
    print(f"Average fraud rate: {high_fraud_cards['card_fraud_rate'].mean()*100:.2f}%")
    print(f"These cards account for {high_fraud_cards['card_txn_count']}")

# Cards with variance
high_variance_cards = card_aggs[card_aggs['card_amt_std'] > card_aggs['card_amt_std'].quantile(0.90)]
print(f"High variance cards (>90th percentile std): {len(high_variance_cards):,}")
print(f"Average std: ${high_variance_cards['card_amt_std'].mean():.2f}")

# Derived features

# 1. Card amount range
train['card_amt_range'] = train['card_amt_max'] - train['card_amt_min']

# 2. Deviation
train['deviation'] = train['TransactionAmt'] - train['card_amt_mean']

# 3. Ratio (current amount : card mean)
train['amt_ratio_card_mean'] = train['TransactionAmt'] / (train['card_amt_mean'] + 0.01)

# Deviation patterns
large_deviation = train['deviation'] > train['deviation'].quantile(0.95)
print(f"\n   Large positive deviation (>95th percentile):")
print(f"      Transactions: {large_deviation.sum():,} ({large_deviation.mean()*100:.2f}%)")
print(f"      Fraud rate: {train[large_deviation]['isFraud'].mean()*100:.2f}%")
print(f"      vs Normal: {train[~large_deviation]['isFraud'].mean()*100:.2f}%")



   card1  card_amt_mean  card_amt_std  card_amt_min  card_amt_max  \
0   1000      23.443000           NaN        23.443        23.443   
1   1001      79.666667     89.494879        27.000       183.000   
2   1004     136.400000     93.577775        30.000       226.000   
3   1005      50.000000           NaN        50.000        50.000   
4   1006     133.333333     28.867513       100.000       150.000   

   card_amt_median  card_txn_count  card_fraud_rate  
0           23.443               1              0.0  
1           29.000               3              0.0  
2          150.000               5              0.0  
3           50.000               1              0.0  
4          150.000               3              0.0  
502
High risk cards: 288
Average fraud rate: 90.69%
These cards account for 39       1
154      7
216      1
292      3
313      1
        ..
13423    1
13435    1
13475    2
13494    1
13507    1
Name: card_txn_count, Length: 288, dtype: int64
High variance ca

In [14]:
# Product Level Aggregations
# Aggregate by product code (PrpductCD)

product_aggs = train.groupby('ProductCD').agg({
    'TransactionAmt': ['mean', 'std', 'median', 'count'],
    'isFraud': 'mean'
}).reset_index()

product_aggs.columns = [
    'ProductCD',
    'product_amt_mean',
    'product_amt_std',
    'product_amt_median',
    'product_txn_count',
    'product_fraud_rate'
]

# Merge back into train data
train = train.merge(product_aggs, on='ProductCD', how='left')

# Derived features
# 1. Product deviation
train['product_deviation'] = train['TransactionAmt'] - train['product_amt_mean']

# 2. Amount ratio to product mean
train['amount_ratio_product_mean'] = train['TransactionAmt'] / (train['product_amt_mean'] + 0.01)

# Analysis
for idx, row in product_aggs.sort_values('product_fraud_rate', ascending=False).iterrows():
    print(f"   Product {row['ProductCD']}: {row['product_fraud_rate']*100:5.2f}% fraud "
          f"({int(row['product_txn_count']):>7,} txns, avg ${row['product_amt_mean']:.2f})")



   Product C: 11.69% fraud ( 68,519 txns, avg $42.87)
   Product S:  5.90% fraud ( 11,628 txns, avg $60.27)
   Product H:  4.77% fraud ( 33,024 txns, avg $73.17)
   Product R:  3.78% fraud ( 37,699 txns, avg $168.31)
   Product W:  2.04% fraud (439,670 txns, avg $153.16)


In [16]:
# Email domain aggregations

# Buyer email domains
email_buyer_aggs = train.groupby('P_emaildomain').agg({
    'TransactionAmt': ['mean', 'count'],
    'isFraud': 'mean'
}).reset_index()

email_buyer_aggs.columns = [
    'P_emaildomain',
    'P_email_amt_mean',
    'P_email_txn_count',
    'P_email_fraud_rate'
]

# Merge back to train
train = train.merge(email_buyer_aggs, on='P_emaildomain', how='left')

# Recipient email domains
email_recipient_aggs = train.groupby('R_emaildomain').agg({
    'TransactionAmt': ['mean', 'count'],
    'isFraud': 'mean'
}).reset_index()

email_recipient_aggs.columns = [
    'R_emaildomain',
    'R_email_amt_mean',
    'R_email_txn_count',
    'R_email_fraud_rate'
]

# Merge back to train
train = train.merge(email_recipient_aggs, on='R_emaildomain', how='left')

# Filter domains with sufficient data
email_buyer_filtered = email_buyer_aggs[email_buyer_aggs['P_email_txn_count'] > 100].copy()
email_buyer_aggs =email_buyer_filtered.sort_values('P_email_fraud_rate', ascending=False)

for idx, row in email_buyer_aggs.head(10).iterrows():
    domain = str(row['P_emaildomain'])[:28] if pd.notna(row['P_emaildomain']) else 'missing'
    print(f"{domain:30s} {int(row['P_email_txn_count']):>10,} {row['P_email_fraud_rate']*100:>9.2f}%")


major_providers = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'aol.com']
for provider in major_providers:
    provider_data = email_buyer_aggs[email_buyer_aggs['P_emaildomain'] == provider]
    if len(provider_data) > 0:
        fraud_rate = provider_data['P_email_fraud_rate'].values[0] * 100
        txn_count = provider_data['P_email_txn_count'].values[0]
        print(f"   {provider:20s}: {fraud_rate:5.2f}% fraud ({txn_count:>7,} txns)")



mail.com                              559     18.96%
outlook.es                            438     13.01%
aim.com                               315     12.70%
outlook.com                         5,096      9.46%
hotmail.es                            305      6.56%
live.com.mx                           749      5.47%
hotmail.com                        45,250      5.30%
gmail.com                         228,355      4.35%
yahoo.fr                              143      3.50%
embarqmail.com                        260      3.46%
   gmail.com           :  4.35% fraud (228,355 txns)
   yahoo.com           :  2.28% fraud (100,934 txns)
   hotmail.com         :  5.30% fraud ( 45,250 txns)
   outlook.com         :  9.46% fraud (  5,096 txns)
   aol.com             :  2.18% fraud ( 28,289 txns)


In [19]:
# Addresses aggregations
# addr1 aggregations
addr1_aggs = train.groupby('addr1').agg({
    'TransactionAmt': ['mean', 'count'],
    'isFraud': 'mean'
}).reset_index()

addr1_aggs.columns = [
    'addr1',
    'addr1_amt_mean',
    'addr1_txn_count',
    'addr1_fraud_rate'
]

# Merge back to train
train = train.merge(addr1_aggs, on='addr1', how='left')

# addr2 aggregations
addr2_aggs = train.groupby('addr2').agg({
    'TransactionAmt': ['mean', 'count'],
    'isFraud': 'mean'
}).reset_index()

addr2_aggs.columns = [
    'addr2',
    'addr2_amt_mean',
    'addr2_txn_count',
    'addr2_fraud_rate'
]

# Merge back to train
train = train.merge(addr2_aggs, on='addr2', how='left')

# addr1 high risk 
addr1_high_risk= addr1_aggs[
    (addr1_aggs['addr1_fraud_rate'] > 0.10) &
    (addr1_aggs['addr1_txn_count'] > 50)
]

print(f"\n   High-risk addr1 (>10% fraud, >50 txns): {len(addr1_high_risk)}")
if len(addr1_high_risk) > 0:
    print(f"      Average fraud rate: {addr1_high_risk['addr1_fraud_rate'].mean()*100:.2f}%")
    print(f"      Total transactions: {addr1_high_risk['addr1_txn_count'].sum():,}")

# addr2 high risk
addr2_high_risk = addr2_aggs[
    (addr2_aggs['addr2_fraud_rate'] > 0.10) & 
    (addr2_aggs['addr2_txn_count'] > 50)
]

print(f"\n   High-risk addr2 (>10% fraud, >50 txns): {len(addr2_high_risk)}")
if len(addr2_high_risk) > 0:
    print(f"      Average fraud rate: {addr2_high_risk['addr2_fraud_rate'].mean()*100:.2f}%")
    print(f"      Total transactions: {addr2_high_risk['addr2_txn_count'].sum():,}")




   High-risk addr1 (>10% fraud, >50 txns): 5
      Average fraud rate: 18.86%
      Total transactions: 1,241

   High-risk addr2 (>10% fraud, >50 txns): 2
      Average fraud rate: 33.80%
      Total transactions: 720
